# 00 · The Workshop – set up Stark's lab

> *"Sometimes you gotta run before you can walk."* – Tony Stark

Every hero starts in a garage. Ours is a **Microsoft Foundry** project: one place for models, agents, data, evaluations and monitoring.

| 🧱 Piece | 🔎 What it is | 🏷️ Name in this repo |
|---|---|---|
| **Foundry resource** | The Azure resource (`Microsoft.CognitiveServices/accounts`, kind `AIServices`). Holds model deployments and the Foundry Tools (Document Intelligence, Content Understanding, ...). | `STARK_FOUNDRY_ACCOUNT` |
| **Foundry project** | A workspace inside the resource. Agents, conversations, datasets and evaluations live here. | `STARK_FOUNDRY_PROJECT` |
| **Project endpoint** | The one URL the SDK needs: `https://<resource>.services.ai.azure.com/api/projects/<project>` | `STARK_PROJECT_ENDPOINT` |
| **Model deployment** | A named, billable instance of a model. You call the *deployment name*, not the model name. | `STARK_CHAT_MODEL`, ... |
| **Application Insights** | Connected to the project: traces and the agent monitoring dashboard. | `STARK_APPINSIGHTS` |

**Deploy first** (once, about 10 minutes): `azd up`. It provisions everything above and writes `.env` for the notebooks. Details: [README](../README.md#-deploy-with-azd) and [docs/architecture.md](../docs/architecture.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

env = check_env()          # fails with a hint if .env is missing
for k, v in env.items():
    print(f"{k:28} {v}")

## Who am I?
The SDK uses `DefaultAzureCredential`: it picks up your `az login` (or `azd auth login`) session. There are **no API keys** in this repo – the Foundry resource has local auth disabled and your Entra identity holds the *Azure AI User* role.

In [ ]:
import subprocess, shutil, json
az = shutil.which("az") or "az"
who = json.loads(subprocess.run([az, "account", "show", "-o", "json"], capture_output=True, text=True).stdout or "{}")
print("user        :", (who.get("user") or {}).get("name"))
print("subscription:", who.get("name"))

## What did `azd up` deploy?
The project client can list the deployments of its Foundry resource. These are the models every later chapter calls.

In [ ]:
for d in project().deployments.list():
    print(f"{d.name:26} {getattr(d, 'model_publisher', '')}/{getattr(d, 'model_name', '')}  v{getattr(d, 'model_version', '')}")

## Smoke test
One call proves endpoint, identity and a model deployment all work. Chapter 01 explains this call line by line.

In [ ]:
oai = openai_client()
r = oai.responses.create(model=model("fast"), input="Say 'Workshop online' and nothing else.")
print(r.output_text)

## Portal tour (2 minutes)
Open <https://ai.azure.com>, switch on **New Foundry**, pick the `stark-workshop` project and find:

1. **Overview** – the project endpoint you just used.
2. **Build → Models** – the deployments listed above.
3. **Build → Agents** – empty for now. Chapter 04 fills it.
4. **Traces / Monitor** – fills up after chapter 03.

> 💡 Next: [01 · Dummy's first call](01_dummy_first_call.ipynb)